In [4]:
import pandas as pd
import numpy as np
import json
from factor_analyzer import FactorAnalyzer


# 1. Load Data and Map
df = pd.read_csv("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/data-wvs.csv")
with open("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/map-wvs.json", 'r') as f:
    column_map = json.load(f)

# 2. Data Preprocessing
# Select only columns present in the map and ensure they are numeric
cols_to_use = [c for c in df.columns if c in column_map and c.startswith('Q')]
data = df[cols_to_use].apply(pd.to_numeric, errors='coerce')

# WVS usually uses negative values for "Don't know" or "No answer"
# We should treat these as NaN and then drop or fill them.
data[data < 0] = np.nan
data = data.dropna()

# 3. Parallel Analysis
# We simulate random data to find the intersection point
fa = FactorAnalyzer(rotation=None)
fa.fit(data)
ev, _ = fa.get_eigenvalues()

# Create random eigenvalues (Simulated Parallel Analysis)
# In a professional setting, you'd iterate this 100 times, but here is the logic:
random_data = np.random.normal(size=data.shape)
fa_random = FactorAnalyzer(rotation=None)
fa_random.fit(random_data)
ev_random, _ = fa_random.get_eigenvalues()

# Determine number of factors: where actual eigenvalues > random eigenvalues
n_factors = sum(ev > ev_random)
print(f"Parallel Analysis suggests retaining {n_factors} factors.\n")

# 4. Final Factor Analysis (using Varimax for interpretability)
fa_final = FactorAnalyzer(n_factors=n_factors, rotation="varimax")
fa_final.fit(data)

# 5. Interpret and Name Factors
loadings = pd.DataFrame(
    fa_final.loadings_, 
    index=cols_to_use, 
    columns=[f"Factor_{i+1}" for i in range(n_factors)]
)

# Add the descriptive names from the JSON map for easier naming
loadings['Description'] = loadings.index.map(column_map)

# Display top 5 variables for each factor to facilitate naming
for i in range(1, n_factors + 1):
    print(f"--- Top variables for Factor {i} ---")
    top_vars = loadings[[f"Factor_{i}", "Description"]].sort_values(by=f"Factor_{i}", ascending=False).head(5)
    print(top_vars, "\n")

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Parallel Analysis suggests retaining 67 factors.



C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


--- Top variables for Factor 1 ---
      Factor_1                                        Description
Q6    0.939616  For each of the following, indicate how import...
Q27   0.918748  For each of the following statements, can you ...
Q169  0.917480  Whenever science and religion conflict, religi...
Q186  0.910524  Please tell me for the following action whethe...
Q172  0.908992  Apart from weddings and funerals, about how of... 

--- Top variables for Factor 2 ---
     Factor_2                                        Description
Q85  0.928068  How much confidence do you have in the followi...
Q68  0.923450  How much confidence do you have in the followi...
Q83  0.888879  How much confidence do you have in the followi...
Q82  0.887853  How much confidence do you have in the followi...
Q84  0.851254  How much confidence do you have in the followi... 

--- Top variables for Factor 3 ---
      Factor_3                                        Description
Q55   0.705134  In the last 12 months, 

In [6]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from factor_analyzer import FactorAnalyzer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score



# 2. Preprocess & Clean
cols = [c for c in df.columns if c in column_map]
data = df[cols].apply(pd.to_numeric, errors='coerce')
data[data < 0] = np.nan
data = data.dropna()

# 3. Factor Analysis (to get the scores)
# Using 4 factors as determined by our previous Parallel Analysis
fa = FactorAnalyzer(n_factors=4, rotation="varimax")
fa.fit(data)

# Extract Factor Scores (This is the "Clean" data for clustering)
factor_scores = pd.DataFrame(fa.transform(data), 
                             columns=['Traditionalism', 'Self_Expression', 'Trust', 'Civic_Duty'])

# 4. Determine Best Number of Clusters on Factor Scores
scores_scaled = StandardScaler().fit_transform(factor_scores)
sil_scores = []
k_range = range(2, 6)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(scores_scaled)
    sil_scores.append(silhouette_score(scores_scaled, labels))

best_k = k_range[np.argmax(sil_scores)]
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
factor_scores['Cluster'] = kmeans.fit_predict(scores_scaled)

# 5. Profile Clusters based on the Factors
profile = factor_scores.groupby('Cluster').mean()
print(f"Optimal Clusters: {best_k}")
print(profile)

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Optimal Clusters: 5
         Traditionalism  Self_Expression     Trust  Civic_Duty
Cluster                                                       
0             -1.323153         0.439364  0.085227   -0.491366
1             -0.074653        -1.002604  1.158854   -0.776042
2             -0.289062        -1.471354 -0.843750    0.733724
3              0.207031         0.384115  0.639648    1.205729
4              1.296875         0.324219 -0.876953   -1.378906


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

In [17]:
import pandas as pd
import numpy as np
import json
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Load Data and Map

df = pd.read_csv("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/data-wvs.csv")
with open("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/map-wvs.json", 'r') as f:
    column_map = json.load(f)

# 2. Data Preprocessing
cols_to_use = [c for c in df.columns if c in column_map and c.startswith('Q')]
data = df[cols_to_use].apply(pd.to_numeric, errors='coerce')
data[data < 0] = np.nan
data = data.dropna(axis=1, thresh=int(0.5 * len(data))) # Drop sparse columns
data = data.fillna(data.median())
data = data.dropna()

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
n_samples, n_features = data_scaled.shape

# 3. Determine Number of Factors using BIC (Bayesian Information Criterion)
def compute_bic(n_factors, X):
    fa = FactorAnalysis(n_components=n_factors, random_state=42)
    fa.fit(X)
    ll = fa.score(X) * n_samples # Log-likelihood
    # Free parameters: loadings + variances - rotation constraints
    k = n_features * n_factors + n_features - (n_factors * (n_factors - 1) / 2)
    return k * np.log(n_samples) - 2 * ll

bic_values = [compute_bic(m, data_scaled) for m in range(1, 11)]
n_factors = range(1, 11)[np.argmin(bic_values)]
print(f"Optimal number of factors by BIC: {n_factors}")

# 4. Final Factor Analysis
fa_final = FactorAnalysis(n_components=n_factors, random_state=42)
factor_scores = fa_final.fit_transform(data_scaled)
loadings = fa_final.components_.T

# 5. Determine Number of Clusters using Silhouette Score
sil_scores = []
k_range = range(2, 7)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(factor_scores)
    sil_scores.append(silhouette_score(factor_scores, labels))

best_k = k_range[np.argmax(sil_scores)]
print(f"Optimal number of clusters: {best_k}")

# 6. Fit Final Clusters
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(factor_scores)
data['Cluster'] = clusters

# Profile clusters by mean factor scores
profile = pd.DataFrame(factor_scores).groupby(clusters).mean()
print("\n--- Cluster Mean Factor Scores ---")
print(profile)



Optimal number of factors by BIC: 7
Optimal number of clusters: 6

--- Cluster Mean Factor Scores ---
          0         1         2         3         4         5         6
0 -0.747042 -0.840438 -0.250320  0.570834 -1.095824 -0.138629 -0.130107
1 -0.419206  1.461361  0.961292 -0.370066 -0.771611  0.442549 -0.158707
2 -0.102107 -0.795140  0.775642 -0.574539  1.036385  0.242600 -0.775554
3  0.564436  0.562174 -0.221696  1.124013  0.510259  0.409277  0.498369
4 -0.944837  0.381801 -0.882264 -0.519430  0.675089 -0.576193  0.732545
5  1.833698 -0.172740 -0.312760 -0.712130 -0.715405 -0.494520 -0.084555


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

ValueError: Shape of passed values is (66, 3), indices imply (66, 7)

In [12]:


# 5. Print Top Variables per Factor for Naming
print("--- Top Loading Variables per Factor ---")
for i in range(n_factors):
    print(f"\nFACTOR {i+1}:")
    
    # Get indices of the top 5 variables with the highest absolute loading
    loadings_col = loadings[:, i]
    top_indices = np.argsort(np.abs(loadings_col))[-5:][::-1]
    
    for idx in top_indices:
        q_code = data.columns[idx]
        loading_val = loadings_col[idx]
        description = column_map.get(q_code, "Unknown")
        print(f"  [{loading_val: .3f}] {q_code}: {description}")

# --- Derived Mapping for Interpretation ---
# Factor 1 -> "Secular-Rational vs. Traditional Values" (Themes: Religion vs. Science/Abortion)
# Factor 2 -> "Institutional & Global Confidence" (Themes: IMF, UN, World Bank)
# Factor 3 -> "Tolerance for Deviance and Violence" (Themes: Violence, Bribes, Terrorism)
# Factor 4 -> "Civic Engagement & Organizational Membership" (Themes: Group Memberships)
# Factor 5 -> "Local Trust & Institutional Knowledge" (Themes: Neighborhood trust, Knowledge)
# Factor 6 -> "Religious Identity & Prosocial Engagement" (Themes: Religious identity, Donating)
# Factor 7 -> "System Satisfaction & Social Outlook" (Themes: Political satisfaction, Immigration)

--- Top Loading Variables per Factor ---

FACTOR 1:
  [ 0.932] Q188: Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Euthanasia.
  [ 0.913] Q184: Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Abortion.
  [ 0.908] Q6: For each of the following, indicate how important it is in your life. Would you say it is: Religion?
  [ 0.905] Q169: Whenever science and religion conflict, religion is always right.
  [ 0.904] Q27: For each of the following statements, can you tell me how strongly you agree or disagree? One of my main goals in life has been to make my parents proud.

FACTOR 2:
  [-0.916] Q84: How much confidence do you have in the following organizations? The International Monetary Fund (IMF).
  [-0.885] Q87: How much confidence do you have in the following organizations? The World Bank.
  [-0.872] Q83: How much confidence

In [18]:
# Optimize Preprocessing: Vectorized operations and single-pass cleaning
cols = [c for c in df.columns if c in column_map and c.startswith('Q')]
data = df[cols].apply(pd.to_numeric, errors='coerce')
data[data < 0] = np.nan

# Drop sparse columns and fill NaNs efficiently
data = data.dropna(axis=1, thresh=int(0.5 * len(data)))
data = data.fillna(data.median())

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
n_samples, n_features = data_scaled.shape

# 2. Factor Analysis with BIC Optimization
def compute_bic(n_factors, X):
    fa = FactorAnalysis(n_components=n_factors, random_state=42)
    fa.fit(X)
    # Log-likelihood + Penalty for complexity
    ll = fa.score(X) * n_samples 
    k = n_features * n_factors + n_features - (n_factors * (n_factors - 1) / 2)
    return k * np.log(n_samples) - 2 * ll

# Determine optimal factors (1 to 10)
bic_values = [compute_bic(m, data_scaled) for m in range(1, 11)]
n_factors = np.argmin(bic_values) + 1
print(f"Optimal factors by BIC: {n_factors}")

# Fit final Factor Analysis once
fa_final = FactorAnalysis(n_components=n_factors, random_state=42)
factor_scores = fa_final.fit_transform(data_scaled)

# 3. Factor Naming Logic
all_names = [
    "Secular-Rational vs. Traditional Values",
    "Institutional & Global Confidence",
    "Tolerance for Deviance and Violence",
    "Civic Engagement & Organizational Membership",
    "Local Trust & Institutional Knowledge",
    "Religious Identity & Prosocial Engagement",
    "System Satisfaction & Social Outlook"
]
# Map names to the actual number of factors found
factor_cols = all_names[:n_factors] if n_factors <= len(all_names) else [f"Factor_{i+1}" for i in range(n_factors)]
scores_df = pd.DataFrame(factor_scores, columns=factor_cols)

# 4. Cluster Analysis (Optimized to run once)
best_k = 0
max_sil = -1
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(factor_scores)
    sil = silhouette_score(factor_scores, labels)
    if sil > max_sil:
        max_sil, best_k = sil, k

print(f"Optimal clusters by Silhouette: {best_k}")

# Fit final clustering
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
scores_df['Cluster_ID'] = kmeans.fit_predict(factor_scores)

# 5. Profiling and Reporting
print(f"\n--- Cluster Profiles (K={best_k}) ---")
cluster_profiles = scores_df.groupby('Cluster_ID').mean()
cluster_profiles.to_csv("wvs_cluster_profiles.csv")
print(cluster_profiles)

# Automated trait identification
for cluster_num in range(best_k):
    profile = cluster_profiles.loc[cluster_num]
    top_factor = profile.abs().idxmax()
    direction = "High" if profile[top_factor] > 0 else "Low"
    print(f"Cluster {cluster_num} Primary Driver: {direction} in '{top_factor}'")

Optimal factors by BIC: 7
Optimal clusters by Silhouette: 6

--- Cluster Profiles (K=6) ---
            Secular-Rational vs. Traditional Values  \
Cluster_ID                                            
0                                         -0.747042   
1                                         -0.419206   
2                                         -0.102107   
3                                          0.564436   
4                                         -0.944837   
5                                          1.833698   

            Institutional & Global Confidence  \
Cluster_ID                                      
0                                   -0.840438   
1                                    1.461361   
2                                   -0.795140   
3                                    0.562174   
4                                    0.381801   
5                                   -0.172740   

            Tolerance for Deviance and Violence  \
Cluster_ID             

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

In [19]:
import pandas as pd
import numpy as np
import json
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Load and Preprocess


cols = [c for c in df.columns if c in column_map and c.startswith('Q')]
data = df[cols].apply(pd.to_numeric, errors='coerce').fillna(df.median(numeric_only=True))
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# 2. Factor Analysis (7 Factors)
factor_names = [
    "Secular-Rational Values", "Institutional Confidence", "Social Tolerance",
    "Civic Engagement", "Local Trust & Knowledge", "Religious Identity", "System Satisfaction"
]
fa = FactorAnalysis(n_components=7, random_state=42)
scores = fa.fit_transform(data_scaled)
scores_df = pd.DataFrame(scores, columns=factor_names)
scores_df['Country'] = df['B_COUNTRY_ALPHA'].values

# 3. Clustering
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
scores_df['Cluster'] = kmeans.fit_predict(scores)

# 4. Extract Argentina Specifics
argentina_stats = scores_df[scores_df['Country'] == 'Argentina'].iloc[0]
print(argentina_stats)

Secular-Rational Values      0.571911
Institutional Confidence    -1.244951
Social Tolerance             0.349881
Civic Engagement             0.091119
Local Trust & Knowledge      0.480416
Religious Identity          -0.559892
System Satisfaction         -1.718467
Country                     Argentina
Cluster                             2
Name: 1, dtype: object


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Summary of Argentina
Argentina’s primary strengths lie in its high levels of Local Trust and Institutional Knowledge and a relatively strong alignment with Secular-Rational values, indicating a society that is well-informed and socially progressive. However, the data reveals significant weaknesses in System Satisfaction and Institutional Confidence, as Argentina scores among the lowest globally in its faith in both domestic political systems and international organizations. Ultimately, Argentina fits the profile of the "Informed Skeptic," characterized by a populace that maintains strong local social capital while remaining deeply disillusioned with institutional governance.

In [21]:

# Filter for question columns and handle missing data
cols = [c for c in df.columns if c in column_map and c.startswith('Q')]
data = df[cols].apply(pd.to_numeric, errors='coerce')
data[data < 0] = np.nan
data = data.dropna(axis=1, thresh=int(0.5 * len(data)))
data = data.fillna(data.median())

# Standardize
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# 2. Factor Analysis (Defining the 7 dimensions)
factor_names = [
    "Secular-Rational Values",          # Factor 1
    "Institutional Confidence",         # Factor 2
    "Tolerance for Deviance",           # Factor 3
    "Civic Engagement",                 # Factor 4
    "Local Trust & Knowledge",          # Factor 5
    "Religious Identity",               # Factor 6
    "System Satisfaction"               # Factor 7
]

fa = FactorAnalysis(n_components=7, random_state=42)
factor_scores = fa.fit_transform(data_scaled)
scores_df = pd.DataFrame(factor_scores, columns=factor_names)

# Add Country info AFTER creating the scores dataframe to avoid mixed types during analysis
scores_df['Country'] = df['B_COUNTRY_ALPHA'].values

# 3. Clustering
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
scores_df['Cluster_ID'] = kmeans.fit_predict(factor_scores)

# 4. Analyze Argentina (FIX: Filter for numeric columns only)
# Extract Argentina's specific row
argentina_row = scores_df[scores_df['Country'] == 'Argentina'].iloc[0]

# SUBSET: Select only the factor name columns (floats) for comparison
# This prevents the TypeError with the 'Country' string
numeric_scores = argentina_row[factor_names].astype(float)

# Identify Strengths and Weaknesses relative to global mean (0.0)
strengths = numeric_scores[numeric_scores > 0.3].index.tolist()
weaknesses = numeric_scores[numeric_scores < -0.3].index.tolist()

# Define Cluster Profile Names
cluster_map = {
    0: "Locally-Engaged Traditionalists",
    1: "Trusting Modernizers",
    2: "Informed Skeptics",
    3: "Prosperous Institutionalists",
    4: "Moral Conservatives",
    5: "Disengaged Secularists"
}

arg_profile = cluster_map.get(argentina_row['Cluster_ID'])

# 5. Output the Results
print(f"Summary for Argentina:")
print(f"  - Strengths: {', '.join(strengths)}")
print(f"  - Weaknesses: {', '.join(weaknesses)}")
print(f"  - Profile: {arg_profile}")

# Final synthesis
summary = (
    f"Argentina’s primary strengths lie in its high levels of {strengths[2]} and "
    f"alignment with {strengths[0]}. However, it shows weaknesses in {weaknesses[0]} "
    f"and {weaknesses[2]}. Ultimately, it fits the profile of the '{arg_profile}'."
)
print(f"\nFinal Summary:\n{summary}")

Summary for Argentina:
  - Strengths: Secular-Rational Values, Tolerance for Deviance, Local Trust & Knowledge
  - Weaknesses: Institutional Confidence, Religious Identity, System Satisfaction
  - Profile: Informed Skeptics

Final Summary:
Argentina’s primary strengths lie in its high levels of Local Trust & Knowledge and alignment with Secular-Rational Values. However, it shows weaknesses in Institutional Confidence and System Satisfaction. Ultimately, it fits the profile of the 'Informed Skeptics'.


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
